# 00 – DATA COLLECTION
## Electronics Sales Dataset (B-Tech Style)

Mục tiêu:
1. Thu thập dữ liệu gốc từ Kaggle.
2. Kiểm tra tính toàn vẹn của các file CSV.
3. Thống kê quy mô từng bảng.
4. Ghi nhận nguồn dữ liệu và thông tin giấy phép.
5. Phân loại các bảng theo chức năng nghiệp vụ.
6. Xác định các khóa ứng viên và quan hệ dự kiến.

Nguồn: Kaggle – Electronics Sales Dataset (B-Tech Style).

Nguyên tắc:
- Không chỉnh sửa dữ liệu Raw.
- Không xóa bản ghi trùng.
- Không xử lý giá trị thiếu.
- Không thay đổi kiểu dữ liệu gốc.
- Lưu thông tin để các notebook tiếp theo sử dụng.



Cell 1 – Khởi tạo đường dẫn và thư viện

In [2]:
from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
import os
import re
import json
import warnings

warnings.filterwarnings("ignore")

# Tìm thư mục gốc của project, dù mở notebook từ root hay notebooks/
CURRENT_DIR = Path.cwd().resolve()

candidates = [CURRENT_DIR, *CURRENT_DIR.parents]

# VS Code thường khai báo thư mục workspace qua biến môi trường này.
for env_name in ("VSCODE_CWD", "GITHUB_WORKSPACE"):
    env_dir = os.environ.get(env_name)
    if env_dir:
        candidates.append(Path(env_dir).resolve())

PROJECT_DIR = next(
    (candidate for candidate in candidates if (candidate / "data" / "raw").is_dir()),
    None,
)

if PROJECT_DIR is None:
    raise FileNotFoundError(
        "Không tìm thấy thư mục project chứa data/raw/. "
        "Hãy mở VS Code tại thư mục SKU-Weekly-Sales-Forecasting "
        "hoặc đặt kernel working directory về thư mục project. "
        f"Thư mục hiện tại: {CURRENT_DIR}"
    )

RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
REPORT_DIR = PROJECT_DIR / "reports"
PLOT_DIR = PROJECT_DIR / "plots"

for folder in [RAW_DIR, PROCESSED_DIR, REPORT_DIR, PLOT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

EXPECTED_CSV_COUNT = 27

print("Project:", PROJECT_DIR)
print("Raw data:", RAW_DIR)
print("Reports:", REPORT_DIR)

Project: C:\Users\admin\Downloads\important\SKU-Weekly-Sales-Forecasting
Raw data: C:\Users\admin\Downloads\important\SKU-Weekly-Sales-Forecasting\data\raw
Reports: C:\Users\admin\Downloads\important\SKU-Weekly-Sales-Forecasting\reports


Cell 2 – Kiểm tra và đọc thông tin 27 file CSV

In [3]:
def read_csv_safely(file_path):
    """Đọc CSV với một số encoding phổ biến."""
    encodings = ["utf-8-sig", "utf-8", "cp1252", "latin1"]

    last_error = None

    for encoding in encodings:
        try:
            return pd.read_csv(
                file_path,
                encoding=encoding,
                low_memory=False
            )
        except Exception as error:
            last_error = error

    raise last_error


csv_files = sorted(RAW_DIR.glob("*.csv"))

if not csv_files:
    raise FileNotFoundError(
        f"Không tìm thấy CSV trong {RAW_DIR}. "
        "Hãy chép 27 file CSV gốc vào data/raw/ trước khi chạy."
    )

print(f"Số file CSV tìm thấy: {len(csv_files)}")

if len(csv_files) != EXPECTED_CSV_COUNT:
    print(
        f"CẢNH BÁO: Yêu cầu dự kiến {EXPECTED_CSV_COUNT} file, "
        f"nhưng hiện tìm thấy {len(csv_files)} file."
    )

inventory_rows = []
tables = {}

for file_path in csv_files:
    try:
        df = read_csv_safely(file_path)
        tables[file_path.stem] = df

        inventory_rows.append({
            "file_name": file_path.name,
            "table_name": file_path.stem,
            "file_path": str(file_path.relative_to(PROJECT_DIR)),
            "file_size_MB": round(file_path.stat().st_size / (1024 ** 2), 4),
            "row_count": len(df),
            "column_count": len(df.columns),
            "duplicate_rows": int(df.duplicated().sum()),
            "read_status": "SUCCESS",
            "error": ""
        })

    except Exception as error:
        inventory_rows.append({
            "file_name": file_path.name,
            "table_name": file_path.stem,
            "file_path": str(file_path.relative_to(PROJECT_DIR)),
            "file_size_MB": round(file_path.stat().st_size / (1024 ** 2), 4),
            "row_count": None,
            "column_count": None,
            "duplicate_rows": None,
            "read_status": "FAILED",
            "error": str(error)
        })

inventory_df = pd.DataFrame(inventory_rows)

display(inventory_df)
print("Số bảng đọc thành công:", len(tables))

Số file CSV tìm thấy: 27


,file_name,table_name,file_path,file_size_MB,row_count,column_count,duplicate_rows,read_status,error
0,Brands.csv,Brands,data\raw\Brands.csv,0.0012,66,2,0,SUCCESS,
1,BrandSubcategoryMap.csv,BrandSubcategoryMap,data\raw\BrandSubcategoryMap.csv,0.0068,310,2,0,SUCCESS,
2,Categories.csv,Categories,data\raw\Categories.csv,0.0003,8,2,0,SUCCESS,
3,CustomerReturns.csv,CustomerReturns,data\raw\CustomerReturns.csv,2.3463,23790,11,25,SUCCESS,
4,Customers.csv,Customers,data\raw\Customers.csv,8.0068,140368,5,368,SUCCESS,
5,Governorates.csv,Governorates,data\raw\Governorates.csv,0.0006,27,2,0,SUCCESS,
6,InventoryBalances.csv,InventoryBalances,data\raw\InventoryBalances.csv,0.5410,12104,9,0,SUCCESS,
7,InventoryMovements.csv,InventoryMovements,data\raw\InventoryMovements.csv,98.9265,961048,15,0,SUCCESS,
8,Locations.csv,Locations,data\raw\Locations.csv,0.0017,23,8,0,SUCCESS,
9,PaymentMethods.csv,PaymentMethods,data\raw\PaymentMethods.csv,0.0003,5,3,0,SUCCESS,


Số bảng đọc thành công: 27


Cell 3 – Phân loại bảng theo tên và tìm khóa ứng viên

In [4]:
def classify_table(table_name):
    name = table_name.lower()

    rules = {
        "Sales": ["sales", "order", "transaction", "invoice"],
        "Product": ["product", "variant", "sku", "category", "brand"],
        "Inventory": ["inventory", "stock", "warehouse"],
        "Customer": ["customer", "client"],
        "Location": ["location", "store", "branch", "warehouse"],
        "Promotion": ["promotion", "discount", "campaign"],
        "Other": []
    }

    for category, keywords in rules.items():
        if any(keyword in name for keyword in keywords):
            return category

    return "Other"


inventory_df["business_group"] = inventory_df["table_name"].apply(
    classify_table
)

# Chuẩn hóa tên cột để tìm cột chung giữa các bảng
def normalize_column_name(column):
    return re.sub(r"[^a-z0-9]", "", str(column).lower())


relationship_rows = []

table_columns = {
    table_name: {
        normalize_column_name(col): col
        for col in df.columns
    }
    for table_name, df in tables.items()
}

table_names = list(tables.keys())

for i in range(len(table_names)):
    for j in range(i + 1, len(table_names)):
        table_a = table_names[i]
        table_b = table_names[j]

        cols_a = table_columns[table_a]
        cols_b = table_columns[table_b]

        common_cols = sorted(set(cols_a) & set(cols_b))

        for normalized_col in common_cols:
            col_a = cols_a[normalized_col]
            col_b = cols_b[normalized_col]

            df_a = tables[table_a]
            df_b = tables[table_b]

            non_null_a = df_a[col_a].dropna()
            non_null_b = df_b[col_b].dropna()

            unique_a = non_null_a.nunique()
            unique_b = non_null_b.nunique()

            relationship_rows.append({
                "table_left": table_a,
                "column_left": col_a,
                "table_right": table_b,
                "column_right": col_b,
                "left_unique_rate": (
                    unique_a / len(non_null_a) if len(non_null_a) else np.nan
                ),
                "right_unique_rate": (
                    unique_b / len(non_null_b) if len(non_null_b) else np.nan
                ),
                "left_null_rate": df_a[col_a].isna().mean(),
                "right_null_rate": df_b[col_b].isna().mean(),
                "left_match_rate": (
                    non_null_a.astype(str).isin(
                        set(non_null_b.astype(str))
                    ).mean() if len(non_null_a) else np.nan
                )
            })

relationships_df = pd.DataFrame(relationship_rows)

display(inventory_df[["table_name", "business_group"]])

if not relationships_df.empty:
    display(
        relationships_df.sort_values(
            ["left_match_rate", "left_unique_rate"],
            ascending=False
        )
    )
else:
    print("Chưa tìm thấy cột trùng tên giữa các bảng.")

,table_name,business_group
0,Brands,Product
1,BrandSubcategoryMap,Product
2,Categories,Other
3,CustomerReturns,Customer
4,Customers,Customer
5,Governorates,Other
6,InventoryBalances,Inventory
7,InventoryMovements,Inventory
8,Locations,Location
9,PaymentMethods,Other


,table_left,column_left,table_right,column_right,left_unique_rate,right_unique_rate,left_null_rate,right_null_rate,left_match_rate
0,Brands,BrandID,BrandSubcategoryMap,BrandID,1.000000,0.212903,0.000000,0.000000,1.000000
1,Brands,BrandID,Products,BrandID,1.000000,0.114583,0.000000,0.000000,1.000000
5,Categories,CategoryID,Subcategories,CategoryID,1.000000,0.123077,0.000000,0.000000,1.000000
22,InventoryMovements,PurchaseReceiptLineID,PurchaseReceiptLines,PurchaseReceiptLineID,1.000000,1.000000,0.933064,0.000000,1.000000
23,InventoryMovements,PurchaseReturnID,PurchaseReturns,PurchaseReturnID,1.000000,1.000000,0.999082,0.000000,1.000000
25,InventoryMovements,SalesOrderLineID,SalesOrderLines,SalesOrderLineID,1.000000,1.000000,0.421367,0.000000,1.000000
28,PaymentMethods,PaymentMethodID,Payments,PaymentMethodID,1.000000,0.000016,0.000000,0.000000,1.000000
29,Payments,SalesOrderID,SalesOrderLines,SalesOrderID,1.000000,0.555120,0.000000,0.000000,1.000000
30,Payments,SalesOrderID,SalesOrders,SalesOrderID,1.000000,0.997380,0.000000,0.000000,1.000000
36,ProductVariants,SKU_ID,SKUSuppliers,SKU_ID,1.000000,0.913880,0.000000,0.000000,1.000000


Cell 4 – Xuất Data Inventory và danh sách quan hệ ứng viên

In [5]:
inventory_path = REPORT_DIR / "data_inventory.xlsx"
relationship_path = REPORT_DIR / "relationship_candidates.xlsx"

with pd.ExcelWriter(inventory_path, engine="openpyxl") as writer:
    inventory_df.to_excel(
        writer,
        sheet_name="Data Inventory",
        index=False
    )

    group_summary = (
        inventory_df.groupby("business_group", dropna=False)
        .agg(
            table_count=("table_name", "count"),
            total_rows=("row_count", "sum"),
            total_size_MB=("file_size_MB", "sum")
        )
        .reset_index()
    )

    group_summary.to_excel(
        writer,
        sheet_name="Business Groups",
        index=False
    )

with pd.ExcelWriter(relationship_path, engine="openpyxl") as writer:
    relationships_df.to_excel(
        writer,
        sheet_name="Relationship Candidates",
        index=False
    )

print("Đã xuất:", inventory_path)
print("Đã xuất:", relationship_path)

Đã xuất: C:\Users\admin\Downloads\important\SKU-Weekly-Sales-Forecasting\reports\data_inventory.xlsx
Đã xuất: C:\Users\admin\Downloads\important\SKU-Weekly-Sales-Forecasting\reports\relationship_candidates.xlsx


Cell 5 – Ghi thông tin nguồn dữ liệu và giấy phép

In [6]:
DATASET_SOURCE = (
    "https://www.kaggle.com/datasets/"
    "fahdelatiby/electronics-sales-dataset-b-tech-style"
)

DATASET_NAME = "Electronics Sales Dataset (B-Tech Style)"
ACCESS_DATE = datetime.now().strftime("%Y-%m-%d")

DATASET_LICENSE = "Chưa xác minh - cần kiểm tra trên trang Kaggle"
DOWNLOAD_METHOD = "Tải thủ công từ Kaggle và lưu CSV vào data/raw/"

source_text = f"""
# Data Sources

## Dataset
- Tên: {DATASET_NAME}
- Nguồn: {DATASET_SOURCE}
- Ngày truy cập: {ACCESS_DATE}
- Phương thức tải: {DOWNLOAD_METHOD}
- Giấy phép: {DATASET_LICENSE}

## Quy mô dữ liệu
- Số file CSV tìm thấy: {len(csv_files)}
- Số bảng đọc thành công: {len(tables)}
- Số file dự kiến theo yêu cầu: {EXPECTED_CSV_COUNT}

## Bảo toàn dữ liệu
Các file trong `data/raw/` được giữ nguyên trạng.
Không thực hiện chỉnh sửa, xóa dòng hoặc thay thế giá trị tại giai đoạn thu thập.

## Lưu ý về quyền riêng tư
Cần kiểm tra các trường có thể định danh cá nhân như email, số điện thoại,
địa chỉ hoặc thông tin khách hàng. Chỉ sử dụng trường cần thiết cho mục tiêu
phân tích và tuân thủ điều kiện sử dụng của nguồn dữ liệu.

## Quan hệ giữa các bảng
Xem `relationship_candidates.xlsx`.
Các cột chung trong file này chỉ là khóa ứng viên, cần xác nhận bằng
ngữ nghĩa nghiệp vụ và kiểm tra khóa trước khi JOIN.
"""

source_path = REPORT_DIR / "data_sources.md"
source_path.write_text(source_text, encoding="utf-8")

print("Đã xuất:", source_path)

Đã xuất: C:\Users\admin\Downloads\important\SKU-Weekly-Sales-Forecasting\reports\data_sources.md
